# Experiment 3.2 — Nonlinear temporal interaction probe

Analysis-only notebook. Training is performed by the Slurm array in `scripts/bash_script/SNN_Bash/run_exp_3_2_cpu_array.bash`; this notebook only reads finalized artifacts.

Primary question: **Do nonlinear interactions between adjacent temporal segments contain discriminative information beyond a position-aware Linear decoder?**

Primary representation: **Fixed250**. Primary effect: `Transition - Linear`. Mechanism control: `Transition - Local`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks/artifacts/experiment_3_2_nonlinear_temporal_interaction/low_rank_residual_v1'
RESULTS = pd.read_csv(ROOT / 'experiment_3_2_results.csv')
DELTAS = pd.read_csv(ROOT / 'experiment_3_2_paired_deltas.csv')
SUMMARY = pd.read_csv(ROOT / 'experiment_3_2_summary.csv')
MECHANISM = pd.read_csv(ROOT / 'experiment_3_2_mechanism_summary.csv')
REPRO = pd.read_csv(ROOT / 'experiment_3_2_baseline_reproduction.csv')
PARAMS = pd.read_csv(ROOT / 'experiment_3_2_parameter_counts.csv')
print('Artifact root:', ROOT)


## 1. Baseline reproduction gate

Experiment 3.2 recomputes the 1.3.3 Linear baseline on every split. The finalizer already rejects an absolute per-split BA drift greater than 0.01; inspect the exact paired differences here.

In [ ]:
display(REPRO.sort_values(['representation', 'split_seed']).reset_index(drop=True))
print('max |difference| =', REPRO.abs_difference.max())


## 2. Main decoder summary

In [ ]:
display(SUMMARY.sort_values(['representation', 'decoder']).reset_index(drop=True))
display(PARAMS.sort_values(['representation', 'decoder']).reset_index(drop=True))


## 3. Paired Linear → Transition comparison

Each line is one user-disjoint split. This exposes whether the mean gain is consistent or driven by one split.

In [ ]:
for representation in ('relative10', 'fixed250', 'fixed500'):
    frame = DELTAS[DELTAS.representation == representation].sort_values('split_seed')
    fig, ax = plt.subplots(figsize=(6, 4))
    for row in frame.itertuples(index=False):
        ax.plot([0, 1], [row.linear_test_ba, row.transition_test_ba], marker='o')
    ax.set_xticks([0, 1], ['Linear', 'Transition'])
    ax.set_ylabel('Test balanced accuracy')
    ax.set_title(f'{representation}: paired Linear → Transition')
    ax.grid(axis='y', alpha=0.25)
    plt.show()


## 4. Paired gains vs Linear

In [ ]:
gain_columns = {
    'Local': 'local_minus_linear',
    'Transition': 'transition_minus_linear',
    'Local+Transition': 'local_transition_minus_linear',
}
for representation in ('relative10', 'fixed250', 'fixed500'):
    frame = DELTAS[DELTAS.representation == representation]
    fig, ax = plt.subplots(figsize=(7, 4))
    for x_index, (label, column) in enumerate(gain_columns.items()):
        values = frame[column].to_numpy()
        ax.scatter([x_index] * len(values), values)
    ax.axhline(0.0, linewidth=1)
    ax.set_xticks(range(len(gain_columns)), list(gain_columns))
    ax.set_ylabel('Test BA gain vs Linear')
    ax.set_title(f'{representation}: paired residual gains')
    ax.grid(axis='y', alpha=0.25)
    plt.show()


## 5. Mechanism control: Transition − Local

Positive values support adjacent cross-bin interaction beyond generic within-bin nonlinearity.

In [ ]:
display(MECHANISM.sort_values('representation').reset_index(drop=True))
fig, ax = plt.subplots(figsize=(7, 4))
for x_index, representation in enumerate(('relative10', 'fixed250', 'fixed500')):
    values = DELTAS.loc[DELTAS.representation == representation, 'transition_minus_local'].to_numpy()
    ax.scatter([x_index] * len(values), values)
ax.axhline(0.0, linewidth=1)
ax.set_xticks([0, 1, 2], ['Relative10', 'Fixed250', 'Fixed500'])
ax.set_ylabel('Transition BA − Local BA')
ax.set_title('Mechanism control across user-disjoint splits')
ax.grid(axis='y', alpha=0.25)
plt.show()


## Interpretation guardrails

- Treat `Transition > Linear` plus `Transition > Local` on Fixed250 as the primary temporal-composition evidence.
- If Local and Transition improve similarly, conclude generic nonlinearity helps; do not claim temporal composition specifically.
- Cross-representation gain differences are secondary because bin counts and trainable parameter counts differ.
- Report all five paired split values, not only mean ± SD.